# Phase space grid example

This notebook demonstrates the **psgrid** workflow as an alternative to the
Gaussian-ensemble approach in `example_1.ipynb`.

### The idea

Run one atom per phase-space grid node $(x_0, v_{x0})$ to map out the transfer function
$p(\text{port} \mid x_0, v_{x0})$.  Once the map exists, cloud-averaged quantities follow
from a weighted sum — no further ais++ calls needed.

### What the PSMAP file stores

One row per port per atom:

| field | meaning |
|---|---|
| `initial_positions/velocities` | $(x_0, y_0, z_0, v_{x0}, \ldots)$ of the atom |
| `final_positions/velocities` | mean port position/velocity at detection |
| `states` | internal state of the port |
| `is_interfering` | 1 if the port has two overlapping wavepackets |
| `amp0`, `amp1` | amplitudes of the two wavepackets ($A_1$, $A_2$; `amp1`=0 if solo) |
| `phase_shifts` | $\Delta\phi$ between the two wavepackets, computed in quad precision in C++ |
| `phase_shift_errors` | numerical error estimate on $\Delta\phi$ |
| `path0`, `path1` | path labels |
| `atom_indices` | which atom this row belongs to |

The port probability is $P = A_0^2 + A_1^2 + 2 A_0 A_1 \cos(\Delta\phi + \delta)$
where $\delta$ is any externally applied phase (phase scan, shear, …).
For a solo port $P = A_0^2$ regardless of $\delta$.

**Setup**
```bash
pip install numpy scipy matplotlib h5py jupyter

cd examples/
ais++ -i input-files/PSR_EXAMPLE_PSGRID.aisi -o output-files/PSR_EXAMPLE_PSGRID.h5
```

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')

plt.rcParams.update({
    'font.family'   : 'serif',
    'font.size'     : 9,
    'axes.labelsize': 9,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'legend.frameon': False,
    'savefig.dpi'   : 150,
})

kB     = 1.380649e-23
m_Sr87 = 86.909 * 1.66054e-27

## 1. Load the phase space map

In [ ]:
FNAME = 'output-files/PSR_EXAMPLE_PSGRID.h5'

with h5py.File(FNAME, 'r') as f:
    init_pos   = f['initial_positions'][:]    # (N_ports, 3)
    init_vel   = f['initial_velocities'][:]   # (N_ports, 3)
    final_pos  = f['final_positions'][:]      # (N_ports, 3)  mean port position
    final_vel  = f['final_velocities'][:]     # (N_ports, 3)  mean port velocity
    dphi       = f['phase_shifts'][:]         # (N_ports,)  Δφ, quad-precise from C++
    dphi_err   = f['phase_shift_errors'][:]   # (N_ports,)
    state      = f['states'][:]               # (N_ports,)
    interfering= f['is_interfering'][:]       # (N_ports,)  bool as int
    atom_idx   = f['atom_indices'][:]         # (N_ports,)
    amp0       = f['amp0'][:]                 # (N_ports,)  |c_0|
    amp1       = f['amp1'][:]                 # (N_ports,)  |c_1|, 0 for solo ports
    path0 = np.array([p.decode() if isinstance(p, bytes) else p for p in f['path0'][:]])
    path1 = np.array([p.decode() if isinstance(p, bytes) else p for p in f['path1'][:]])

n_ports = len(dphi)
n_atoms = int(atom_idx.max()) + 1
n_ports_per_atom = np.unique(atom_idx, return_counts=True)[1][0]

print(f'Total ports      : {n_ports:,}')
print(f'Atoms            : {n_atoms:,}')
print(f'Ports / atom     : {n_ports_per_atom}')
print(f'Interfering ports: {interfering.sum()} / {n_ports}')
print(f'Max Δφ error     : {dphi_err.max():.2e} rad')
print()
# Port structure for atom 0
mask0 = atom_idx == 0
print('Ports for atom 0:')
for i in np.where(mask0)[0]:
    flag = 'interfering' if interfering[i] else 'solo'
    print(f'  {path0[i]:4s}/{path1[i]:4s}  state={state[i]}  '
          f'A0={amp0[i]:.3f}  A1={amp1[i]:.3f}  Δφ={dphi[i]:.4f} rad  ({flag})')

## 2. Port probability formula and grid reshape

$$P(\delta) = A_0^2 + A_1^2 + 2 A_0 A_1 \cos(\Delta\phi + \delta)$$

Reshape the flat arrays into `(nx, nvx, n_ports_per_atom)` using the grid ordering
(x outermost, vx innermost) from the ais++ grid constructor.

In [ ]:
def port_prob(amp0, amp1, dphi, interfering, delta=0.):
    """Port probability with an additional phase delta applied."""
    diag  = amp0**2 + amp1**2
    cross = 2 * amp0 * amp1 * np.cos(dphi + delta)
    return diag + interfering * cross

# Initial conditions: same across all ports of one atom
first_port = np.searchsorted(atom_idx, np.arange(n_atoms))
X0_atoms  = init_pos[first_port, 0]
VX0_atoms = init_vel[first_port, 0]

xs  = np.unique(X0_atoms)
vxs = np.unique(VX0_atoms)
nx, nvx = len(xs), len(vxs)
nP = n_ports_per_atom

print(f'Grid: {nx} × {nvx}')
print(f'  x  : {xs[0]*1e6:.1f} … {xs[-1]*1e6:.1f} µm')
print(f'  vx : {vxs[0]*1e3:.2f} … {vxs[-1]*1e3:.2f} mm/s')

def to_grid(arr_1d):
    """(N_ports,) → (nx, nvx, nP)."""
    return arr_1d.reshape(n_atoms, nP).reshape(nx, nvx, nP)

DPHI  = to_grid(dphi)
AMP0  = to_grid(amp0)
AMP1  = to_grid(amp1)
INTER = to_grid(interfering.astype(float))
STATE = to_grid(state)
XF    = to_grid(final_pos[:, 0])   # mean port x_final

X0_grid  = X0_atoms.reshape(nx, nvx)
VX0_grid = VX0_atoms.reshape(nx, nvx)

# Base probabilities (delta=0)
PROB = port_prob(AMP0, AMP1, DPHI, INTER)

# Port labels from atom 0
port_labels = [f'{path0[first_port[0]+p]}/{path1[first_port[0]+p]}'
               for p in range(nP)]
print('Ports:', port_labels)

## 3. Probability maps and sanity check

In [ ]:
extent = [vxs[0]*1e3, vxs[-1]*1e3, xs[0]*1e6, xs[-1]*1e6]

fig, axes = plt.subplots(1, nP, figsize=(3*nP, 3),
                         gridspec_kw=dict(wspace=0.4))
if nP == 1:
    axes = [axes]

for pi, (ax, lbl) in enumerate(zip(axes, port_labels)):
    im = ax.imshow(PROB[:, :, pi], origin='lower', extent=extent,
                   aspect='auto', cmap='viridis', vmin=0, vmax=0.5)
    ax.set_title(f'{lbl}  (state {STATE[0,0,pi]})', fontsize=8)
    ax.set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0:
        ax.set_ylabel(r'$x_0$ [µm]')
    plt.colorbar(im, ax=ax, label=r'$P$')

plt.suptitle(r'$P(x_0, v_{x0})$ per port — PSR MZ ($n=1$, $T=2.225$ s)', y=1.02)
plt.savefig('psmap_prob_maps.png', bbox_inches='tight')
plt.show()

# Sanity check
total = PROB.sum(axis=2)
print(f'Sum over ports: mean={total.mean():.6f}, min={total.min():.6f}, max={total.max():.6f}')
assert np.allclose(total, 1.0, atol=1e-4), 'Probability sum ≠ 1 — check amplitude threshold / path selection'

## 4. Gaussian-cloud-averaged port probabilities

$$P_\text{port} = \sum_{x_0, v_{x0}} w(x_0, v_{x0})\, P(x_0, v_{x0})$$

This replaces a full $10^6$-atom simulation.

In [ ]:
def gaussian_weights(X0, VX0, x_cm=0., vx_cm=0., sigma_x=1e-4, T_trans=1e-9):
    sigma_vx = np.sqrt(kB * T_trans / m_Sr87)
    w = (np.exp(-0.5*((X0  - x_cm )/sigma_x )**2) *
         np.exp(-0.5*((VX0 - vx_cm)/sigma_vx)**2))
    return w / w.sum()

def cloud_probs(PROB_grid, X0, VX0, **cloud_kw):
    """(n_ports_per_atom,) weighted port probabilities for a Gaussian cloud."""
    w = gaussian_weights(X0, VX0, **cloud_kw)    # (nx, nvx)
    return np.einsum('ij,ijp->p', w, PROB_grid)

P_ports = cloud_probs(PROB, X0_grid, VX0_grid)
print('Port probabilities (σ_x=100 µm, T=1 nK, centred):')
for lbl, p in zip(port_labels, P_ports):
    print(f'  {lbl}  →  {p:.4f}')
print(f'  sum  →  {P_ports.sum():.6f}')

## 5. Offline phase scan

Add a global phase $\delta$ to every port's $\Delta\phi$ and recompute
$P(\delta)$ — equivalent to scanning the final-pulse phase of the interferometer,
without re-running ais++.

In [ ]:
w = gaussian_weights(X0_grid, VX0_grid)

phi_scan  = np.linspace(0, 2*np.pi, 200)
P_scan    = np.zeros((len(phi_scan), nP))

for k, delta in enumerate(phi_scan):
    PROB_k     = port_prob(AMP0, AMP1, DPHI, INTER, delta=delta)
    P_scan[k]  = np.einsum('ij,ijp->p', w, PROB_k)

fig, ax = plt.subplots(figsize=(5.5, 3))
for pi, lbl in enumerate(port_labels):
    ls = '-' if STATE[0, 0, pi] == 0 else '--'
    ax.plot(phi_scan/np.pi, P_scan[:, pi], ls=ls, label=lbl)

ax.set_xlabel(r'Added phase $\delta/\pi$')
ax.set_ylabel('Mean port probability')
ax.set_title('Offline phase scan')
ax.set_xlim(0, 2)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig('psmap_phase_scan.png', bbox_inches='tight')
plt.show()

s0 = np.array([STATE[0, 0, pi] == 0 for pi in range(nP)])
P0 = P_scan[:, s0].sum(axis=1)
contrast = (P0.max() - P0.min()) / (P0.max() + P0.min())
print(f'State-0 fringe contrast: {contrast:.4f}')

## 6. Cloud-position scan — no re-simulation

In [ ]:
x_cm_scan = np.linspace(-3e-4, 3e-4, 120)
s0 = np.array([STATE[0, 0, pi] == 0 for pi in range(nP)])

P0_xcm = np.array([
    cloud_probs(PROB, X0_grid, VX0_grid, x_cm=xcm)[s0].sum()
    for xcm in x_cm_scan
])

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(x_cm_scan*1e6, P0_xcm)
ax.set_xlabel(r'Cloud centre $x_\mathrm{cm}$ [µm]')
ax.set_ylabel(r'$P_{s=0}$')
ax.set_title('State-0 probability vs cloud position — zero ais++ calls')
plt.tight_layout()
plt.savefig('psmap_xcm_scan.png', bbox_inches='tight')
plt.show()